# Spatial econometrics lab
This lab is a small, dense-matrix teaching companion. It covers classic and robust OLS-based LM diagnostics, spatial lag 2SLS, spatial Durbin 2SLS, concentrated SAR/SEM/SDM/SARAR maximum likelihood, uncertainty intervals for spatial impacts, and impact decomposition. Synthetic estimates are teaching demonstrations, not evidence that assumptions hold in applied data.

In [ ]:
import sys
sys.path.insert(0, "../implementations/python/spatial-econometrics")
import numpy as np
from diagnostics import spatial_lm_diagnostics
from impacts import spatial_impacts

weights = np.array([
    [0., 1., 0., 0., 0.],
    [1., 0., 1., 0., 0.],
    [0., 1., 0., 1., 0.],
    [0., 0., 1., 0., 1.],
    [0., 0., 0., 1., 0.],
])
x = np.column_stack([np.ones(5), np.arange(5, dtype=float)])
y = np.array([1.0, 2.1, 2.8, 4.2, 4.9])
diagnostics = spatial_lm_diagnostics(y, x, weights)
print("OLS coefficients:", diagnostics.coefficients)
print("LM lag:", diagnostics.lm_lag)
print("LM error:", diagnostics.lm_error)
print("Robust LM lag:", diagnostics.robust_lm_lag)
print("Robust LM error:", diagnostics.robust_lm_error)


## Read the diagnostics carefully

The classic LM-lag and LM-error tests each have an asymptotic chi-square(1) reference under their null. Both can respond to the competing alternative. The Bera-Yoon robust tests adjust for that competing alternative. If the robust results are `None`, the adjustment is not identified for the supplied design and weights. These are diagnostics, not model selection by p-value ranking.

In [ ]:
from spatial_2sls import spatial_lag_2sls
from maximum_likelihood import spatial_lag_mle, spatial_error_mle
rng = np.random.default_rng(20261115)
n = 80
W_demo = np.zeros((n, n))
for i in range(n - 1):
    W_demo[i, i + 1] = W_demo[i + 1, i] = 1.0
W_demo = W_demo / W_demo.sum(axis=1, keepdims=True)
X_demo = np.column_stack([np.ones(n), rng.normal(size=n)])
y_demo = np.linalg.solve(np.eye(n) - 0.25 * W_demo,
                         X_demo @ np.array([1.0, 0.8]) + rng.normal(scale=0.1, size=n))
iv_fit = spatial_lag_2sls(y_demo, X_demo, W_demo, instrument_orders=1)
sar_fit = spatial_lag_mle(y_demo, X_demo, W_demo)
sem_fit = spatial_error_mle(y_demo, X_demo, W_demo)
print('2SLS rho:', iv_fit.rho)
print('SAR ML rho:', sar_fit.spatial_parameter)
print('SEM ML lambda:', sem_fit.spatial_parameter)


## Estimate a spatial lag model by 2SLS

The spatially lagged response is endogenous. This synthetic example uses X and WX as instruments. Their validity depends on the data-generating assumptions; the code does not prove instrument exogeneity or relevance.

## Estimate a spatial Durbin model by 2SLS

The SDM includes both `Wy` and `WX`. The spatially lagged response is endogenous; `X` and included `WX` are exogenous regressors. Excluded higher-order lags `W²X`, `W³X`, ... instrument `Wy`. A row-standardised intercept is redundant under spatial lagging and is removed from the included WX columns. Identification and instrument strength remain empirical questions.

In [ ]:
from spatial_durbin_2sls import spatial_durbin_2sls
rng = np.random.default_rng(91827)
n = 250
W = np.zeros((n, n))
for i in range(n - 1):
    W[i, i + 1] = W[i + 1, i] = 1.0
W = W / W.sum(axis=1, keepdims=True)
X = np.column_stack([np.ones(n), rng.normal(size=n)])
rho_true = 0.24
beta_true = np.array([1.1, 0.85])
gamma_true = np.array([0.0, -0.3])
epsilon = rng.normal(scale=0.03, size=n)
y_sdm = np.linalg.solve(np.eye(n) - rho_true * W,
                        X @ beta_true + (W @ X) @ gamma_true + epsilon)
sdm_fit = spatial_durbin_2sls(y_sdm, X, W, instrument_orders=5)
print('estimated rho:', sdm_fit.rho)
print('estimated beta:', sdm_fit.beta)
print('estimated gamma:', sdm_fit.gamma)


## Understand the decomposition

For covariate r, the impact matrix is `S_r = (I - rho W)^-1 (beta_r I + gamma_r W)`. The average direct effect is the mean diagonal; the average total effect is the mean row sum; indirect is total minus direct. The coefficients above are assumed inputs, not estimates from the preceding OLS diagnostic. In a real analysis, report uncertainty for all three impacts using the fitted model's covariance or simulation draws.

## Questions

1. Why is the spatially lagged dependent variable endogenous in the SAR model?
2. Why can classic LM-lag and LM-error tests both respond to the other alternative?
3. What changes in the impact matrix when `rho` is zero?
4. Why can the direct impact differ from `beta` when `rho` is nonzero?
5. Which modelling choices are encoded by the weights matrix, and how would alternative weights change the results?

## Estimate the spatial Durbin model by maximum likelihood

At each candidate rho, the estimator transforms the response by `I - rho W`, regresses on `[X, WX]`, and adds the log-determinant Jacobian. Spatially lagged columns redundant with X are omitted; with row-standardised weights, the lag of an intercept is exactly redundant.

In [ ]:
from maximum_likelihood import spatial_durbin_mle
sdm_ml_fit = spatial_durbin_mle(y_sdm, X, W, grid_size=161)
print('estimated rho:', sdm_ml_fit.spatial_parameter)
print('estimated beta:', sdm_ml_fit.coefficients)
print('estimated gamma:', sdm_ml_fit.spatial_covariate_coefficients)
print('profile log likelihood:', sdm_ml_fit.log_likelihood)


## Estimate a SARAR / SAC model by joint maximum likelihood

The SARAR model contains both response feedback and spatially correlated innovations. For each candidate pair `(rho, lambda)`, the likelihood applies both spatial filters and includes both log-determinants. The search starts from a two-dimensional grid and alternates one-dimensional refinements. The returned structural fitted value is `rho W y + X beta`; the innovation variance is estimated after filtering.


In [ ]:
from maximum_likelihood import spatial_sarar_mle
rng = np.random.default_rng(20261116)
n = 75
W_sarar = np.zeros((n, n))
for i in range(n - 1):
    W_sarar[i, i + 1] = W_sarar[i + 1, i] = 1.0
W_sarar = W_sarar / W_sarar.sum(axis=1, keepdims=True)
X_sarar = np.column_stack([np.ones(n), rng.normal(size=n)])
rho_true, lambda_true = 0.22, 0.38
innovation = rng.normal(scale=0.12, size=n)
error = np.linalg.solve(np.eye(n) - lambda_true * W_sarar, innovation)
y_sarar = np.linalg.solve(np.eye(n) - rho_true * W_sarar,
                          X_sarar @ np.array([1.2, -0.55]) + error)
sarar_fit = spatial_sarar_mle(y_sarar, X_sarar, W_sarar, grid_size=31)
print('estimated rho:', sarar_fit.spatial_parameter)
print('estimated lambda:', sarar_fit.error_parameter)
print('estimated beta:', sarar_fit.coefficients)
print('innovation variance:', sarar_fit.sigma2_mle)


## Quantify uncertainty in spatial impacts

Supply the joint covariance of `[rho, beta..., gamma...]` from a fitted estimator or another justified source. The routine simulates parameter vectors, recalculates all impact matrices, and reports percentile intervals and Monte Carlo standard deviations. It rejects ill-conditioned spatial systems; it does not estimate the covariance matrix or guarantee that a normal approximation is appropriate.


In [ ]:
from impacts import spatial_impact_uncertainty
# Illustrative covariance only; in applied work use the fitted model's joint covariance.
parameter_covariance = np.diag([0.01, 0.04, 0.04, 0.04, 0.04])
uncertainty = spatial_impact_uncertainty(
    W, sdm_ml_fit.spatial_parameter, sdm_ml_fit.coefficients,
    parameter_covariance, sdm_ml_fit.spatial_covariate_coefficients,
    draws=2000, seed=2026,
)
for index, item in enumerate(uncertainty):
    print(index, 'direct:', item.direct)
    print(index, 'indirect:', item.indirect)
    print(index, 'total:', item.total)


## Simulate a constrained intrinsic-CAR field

The ICAR precision is singular, with one constant null direction per connected component. The simulator removes each null direction by enforcing a separate sum-to-zero constraint. This is a field simulation, not a fitted regression model.

In [ ]:
from icar import icar_sample
A_icar = np.zeros((8, 8))
for i in range(7):
    A_icar[i, i + 1] = A_icar[i + 1, i] = 1.0
icar_fit = icar_sample(A_icar, precision=2.0, rng=42)
print('component count:', icar_fit.n_components)
print('sum-to-zero check:', np.sum(icar_fit.field))
print('log pseudo-determinant:', icar_fit.log_pseudodeterminant)


## Spatial HAC uncertainty for OLS

This adjusts the OLS sandwich covariance for nearby residual cross-products. It does not change OLS coefficients or solve spatial endogeneity. Coordinates and bandwidth use the same projected linear unit.

In [ ]:
from spatial_hac import spatial_hac_ols
rng = np.random.default_rng(20261116)
coords_hac = np.column_stack([np.arange(60, dtype=float) * 1000.0, np.zeros(60)])
X_hac = np.column_stack([np.ones(60), rng.normal(size=60)])
e_hac = rng.normal(scale=0.5, size=60) + 0.25 * np.roll(rng.normal(size=60), 1)
y_hac = X_hac @ np.array([1.0, 0.6]) + e_hac
hac_fit = spatial_hac_ols(y_hac, X_hac, coords_hac, bandwidth=2500.0)
print('OLS coefficients:', hac_fit.coefficients)
print('spatial HAC standard errors:', hac_fit.standard_errors)
print('nearby pairs:', hac_fit.nonzero_off_diagonal_pairs)


## Two-step heteroskedasticity-robust spatial-lag GMM

The first step uses the conventional 2SLS weighting matrix. The second step estimates the instrument-moment covariance from squared first-step residuals. This adjusts IV inference for heteroskedasticity; it does not model spatially autocorrelated structural errors.

In [ ]:
from spatial_gmm import spatial_lag_gmm
rng = np.random.default_rng(20261117)
n_gmm = 100
W_gmm = np.zeros((n_gmm, n_gmm))
for i in range(n_gmm - 1):
    W_gmm[i, i + 1] = W_gmm[i + 1, i] = 1.0
W_gmm = W_gmm / W_gmm.sum(axis=1, keepdims=True)
xg = rng.normal(size=n_gmm)
X_gmm = np.column_stack([np.ones(n_gmm), xg])
u_gmm = rng.normal(size=n_gmm) * (0.12 + 0.12 * np.abs(xg))
y_gmm = np.linalg.solve(np.eye(n_gmm) - 0.25 * W_gmm, X_gmm @ np.array([1.1, 0.7]) + u_gmm)
gmm_fit = spatial_lag_gmm(y_gmm, X_gmm, W_gmm, instrument_orders=2)
print('rho and beta:', gmm_fit.rho, gmm_fit.beta)
print('robust standard errors:', gmm_fit.standard_errors)
print('J statistic and df:', gmm_fit.j_statistic, gmm_fit.j_degrees_of_freedom)


## Kelejian–Prucha spatial-error generalized moments

This example generates iid innovations, filters them through a spatial-error process, estimates the error coefficient from the three KP innovation moments, and then applies feasible GLS. This is the homoskedastic-innovation KP (1999) procedure, not the heteroskedastic extension.

In [ ]:
from spatial_error_gmm import spatial_error_gmm
rng = np.random.default_rng(20261118)
n_error_gmm = 120
W_error_gmm = np.zeros((n_error_gmm, n_error_gmm))
for i in range(n_error_gmm - 1):
    W_error_gmm[i, i + 1] = W_error_gmm[i + 1, i] = 1.0
W_error_gmm = W_error_gmm / W_error_gmm.sum(axis=1, keepdims=True)
X_error_gmm = np.column_stack([np.ones(n_error_gmm), rng.normal(size=n_error_gmm)])
innovation = rng.normal(scale=0.35, size=n_error_gmm)
u_error_gmm = np.linalg.solve(np.eye(n_error_gmm) - 0.38 * W_error_gmm, innovation)
y_error_gmm = X_error_gmm @ np.array([1.25, -0.55]) + u_error_gmm
error_gmm_fit = spatial_error_gmm(y_error_gmm, X_error_gmm, W_error_gmm)
print('estimated spatial-error lambda:', error_gmm_fit.lambda_)
print('feasible-GLS coefficients:', error_gmm_fit.beta)
print('innovation variance:', error_gmm_fit.innovation_variance)
print('three moment residuals:', error_gmm_fit.moment_residuals)
print('model-based GLS standard errors:', error_gmm_fit.standard_errors)

## First-stage relevance of spatial-lag instruments

The first-stage diagnostic regresses Wy on X and the retained spatial lags of X. Compare the partial R-squared and classical partial F with the HC1 robust Wald F. The latter is not a Kleibergen–Paap statistic and does not test instrument exogeneity.

In [ ]:
from instrument_diagnostics import spatial_lag_instrument_diagnostics
first_stage = spatial_lag_instrument_diagnostics(
    y_gmm, X_gmm, W_gmm, instrument_orders=2,
)
print('partial R-squared:', first_stage.partial_r_squared)
print('classical partial F:', first_stage.classical_partial_f)
print('HC1 robust Wald F (not KP rk F):', first_stage.robust_wald_f)
print('excluded instruments:', first_stage.excluded_instruments)
print('first-stage HC1 standard errors:', first_stage.first_stage_standard_errors_hc1)

## Anderson–Rubin test for a candidate spatial-lag coefficient

For a candidate rho, transform the response to y - rho Wy and test whether the excluded spatial-lag instruments explain any remaining variation after X. The classical F relies on iid homoskedastic Gaussian errors; the HC1 Wald chi-square is asymptotic under independent-observation sampling. The null-size argument does not require strong instruments, but instrument validity remains essential. This cell tests a candidate value only; confidence-set inversion is not implemented.

In [ ]:
from spatial_lag_anderson_rubin import spatial_lag_anderson_rubin
ar_fit = spatial_lag_anderson_rubin(y_gmm, X_gmm, W_gmm, 0.25, instrument_orders=2)
print('candidate rho:', ar_fit.rho_null)
print('classical AR F and df:', ar_fit.classical_f, (ar_fit.classical_f_numerator_df, ar_fit.classical_f_denominator_df))
print('HC1 robust AR Wald chi-square and df:', ar_fit.robust_wald_chi2, ar_fit.robust_wald_degrees_of_freedom)
print('excluded instruments:', ar_fit.excluded_instruments)

## Heteroskedastic spatial-error GM moments

This example uses an OLS residual predictor in a spatial-error model with exogenous X. The two zero-diagonal KP (2010) moments accommodate unknown observation-specific innovation variances. The estimator below targets the disturbance coefficient only; it does not calculate the full joint covariance or standard errors.

In [ ]:
from heteroskedastic_spatial_error_gm import heteroskedastic_spatial_error_gm
rng = np.random.default_rng(20261119)
n_hetero = 120
W_hetero = np.zeros((n_hetero, n_hetero))
for i in range(n_hetero - 1):
    W_hetero[i, i + 1] = W_hetero[i + 1, i] = 1.0
W_hetero = W_hetero / W_hetero.sum(axis=1, keepdims=True)
X_hetero = np.column_stack([np.ones(n_hetero), rng.normal(size=n_hetero)])
innovation_hetero = rng.normal(size=n_hetero) * (0.15 + 0.2 * np.abs(X_hetero[:, 1]))
u_hetero = np.linalg.solve(np.eye(n_hetero) - 0.38 * W_hetero, innovation_hetero)
y_hetero = X_hetero @ np.array([1.25, -0.55]) + u_hetero
beta_ols_hetero = np.linalg.lstsq(X_hetero, y_hetero, rcond=None)[0]
u_predictor_hetero = y_hetero - X_hetero @ beta_ols_hetero
hetero_gm_fit = heteroskedastic_spatial_error_gm(u_predictor_hetero, W_hetero)
print('estimated spatial-error lambda:', hetero_gm_fit.lambda_)
print('two moment residuals:', hetero_gm_fit.moment_residuals)
print('moment objective:', hetero_gm_fit.moment_objective)

## Three-step heteroskedastic SARAR GTSLS

This reproduces the three-stage workflow for a common W=M specification: initial spatial 2SLS, heteroskedastic KP disturbance GM, then Cochrane–Orcutt transformed 2SLS. The reported HC1 covariance is conditional on the estimated disturbance coefficient, not the full joint covariance in Kelejian and Prucha (2010).

In [ ]:
from spatial_sarar_gtsls import spatial_sarar_gtsls
sarar_gtsls_fit = spatial_sarar_gtsls(y_hetero, X_hetero, W_hetero, instrument_orders=3)
print('SAR rho:', sarar_gtsls_fit.rho)
print('spatial-error lambda:', sarar_gtsls_fit.lambda_)
print('regression coefficients:', sarar_gtsls_fit.beta)
print('conditional HC1 standard errors:', sarar_gtsls_fit.standard_errors_hc1_conditional)

## Spatial HAC covariance for SAR 2SLS

Coordinates below are synthetic projected planar coordinates in metres. The bandwidth is therefore also expressed in metres. The kernel-weighted sandwich uses the projected instrumented regressors; this changes inference, not the 2SLS point estimate.

In [ ]:
from spatial_2sls_hac import spatial_lag_2sls_hac
coordinates_hac = np.column_stack([np.arange(len(y_gmm), dtype=float) * 1000.0,
                                    np.zeros(len(y_gmm))])
sar_hac_fit = spatial_lag_2sls_hac(y_gmm, X_gmm, W_gmm, coordinates_hac,
                                   bandwidth=5000.0, instrument_orders=2)
print('rho and beta:', sar_hac_fit.coefficients)
print('spatial HAC standard errors:', sar_hac_fit.standard_errors)
print('bandwidth (metres):', sar_hac_fit.bandwidth)
print('nonzero off-diagonal pairs:', sar_hac_fit.nonzero_off_diagonal_pairs)

## Joint uncertainty for heteroskedastic SARAR GTSLS

The estimator reports both the conditional HC1 covariance and the Kelejian–Prucha plug-in joint covariance. The latter accounts for generated first-stage residuals and covariance between the final transformed IV score and the disturbance GM moments. Parameter order is `[beta..., rho, lambda]`; it is asymptotic and uses identity weighting for the two GM moments.


In [ ]:
print('joint parameter order: beta..., SAR rho, spatial-error lambda')
print('joint KP standard errors:', sarar_gtsls_fit.standard_errors_joint_kp)
print('conditional HC1 standard errors:', sarar_gtsls_fit.standard_errors_hc1_conditional)


## Global Moran's I with permutation inference

This example tests global spatial autocorrelation against a conditional randomization null. Values are permuted across fixed locations while the supplied weights remain unchanged. The two-sided alternative compares deviations from `-1/(n-1)`; one-sided alternatives test greater or lesser statistics. The plus-one correction prevents zero Monte Carlo p-values.

In [ ]:
from morans_i_permutation import morans_i_permutation
rng_moran = np.random.default_rng(20261120)
n_moran = 80
W_moran = np.zeros((n_moran, n_moran))
for i in range(n_moran - 1):
    W_moran[i, i + 1] = W_moran[i + 1, i] = 1.0
W_moran = W_moran / W_moran.sum(axis=1, keepdims=True)
values_moran = rng_moran.normal(size=n_moran)
moran_result = morans_i_permutation(values_moran, W_moran, permutations=999,
                                    alternative='two-sided', seed=20261120)
print('Moran I:', moran_result.statistic)
print('randomization expectation:', moran_result.expected_value)
print('permutation p-value:', moran_result.p_value)
print('permutation mean / SD:', moran_result.permutation_mean,
      moran_result.permutation_standard_deviation)


## Local Moran's I and local cluster classification

The conditional randomization test fixes each focal value, permutes the remaining observations over other locations, and keeps the focal weights row fixed. The example uses Benjamini–Hochberg adjusted p-values across non-island locations. HH/LL/HL/LH labels require adjusted significance; they are not causal explanations.

In [ ]:
from local_morans_i_permutation import local_morans_i_permutation
local_result = local_morans_i_permutation(values_moran, W_moran, permutations=499,
                                          adjustment='bh', alpha=0.05,
                                          alternative='two-sided', seed=20261121)
for label in ('HH', 'LL', 'HL', 'LH', 'not-significant', 'island'):
    print(label, local_result.cluster_labels.count(label))
print('first ten local I:', np.round(local_result.local_i[:10], 3))
print('first ten adjusted p-values:', np.round(local_result.adjusted_p_values[:10], 3))


## Heteroskedasticity diagnostics: Koenker BP and White

These tests diagnose non-constant conditional variance in an OLS baseline. They do not test spatial autocorrelation. White's auxiliary design adds squares and cross-products; rank reduction handles redundant terms.

In [ ]:
from heteroskedasticity_diagnostics import heteroskedasticity_diagnostics
hetero_result = heteroskedasticity_diagnostics(y_sarar, X_sarar)
print('Koenker studentized BP:', hetero_result.koenker_bp)
print('White general test:', hetero_result.white)


## Ramsey RESET: functional-form specification

RESET adds powers of restricted-model fitted values and tests their joint contribution. It is a generic functional-form diagnostic, not a spatial-dependence test.

In [ ]:
from ramsey_reset import ramsey_reset_test
reset_result = ramsey_reset_test(y_sarar, X_sarar, power=3)
print('Ramsey RESET:', reset_result)


## Profile-likelihood interval for the SAR spatial parameter

This inverts a likelihood-ratio test for the spatial lag coefficient. The default cutoff is the 95% chi-square(1) quantile. Endpoints may be clipped by the conservative nonsingularity interval; inspect the truncation flags.

In [ ]:
from spatial_mle_inference import spatial_parameter_profile_interval
sar_interval = spatial_parameter_profile_interval(y_demo, X_demo, W_demo, model='SAR')
print('estimate and LR interval:', sar_interval.estimate, sar_interval.lower, sar_interval.upper)
print('truncated endpoints:', sar_interval.lower_truncated, sar_interval.upper_truncated)


## Invert the spatial-lag Anderson–Rubin test\n\nThe confidence set can be disconnected or extend beyond the searched range. This example uses a caller-supplied chi-square(1) cutoff for the HC1 robust test; expand the rho grid if a component touches an edge.

In [ ]:
from spatial_lag_ar_confidence import spatial_lag_ar_confidence_set
ar_set = spatial_lag_ar_confidence_set(
    y_gmm, X_gmm, W_gmm, np.linspace(-0.8, 0.8, 81),
    critical_value=3.841458820694124, test='robust', instrument_orders=2,
)
print('AR confidence-set segments:', ar_set.segments)
print('grid-limited components:', [s for s in ar_set.segments if s.lower_grid_limited or s.upper_grid_limited])


## Analytic delta-method uncertainty for spatial impacts

This uses the exact Jacobian of the SAR/SDM impact matrix and the joint covariance ordered as [rho, beta..., gamma...]. It is a local normal approximation; compare with Monte Carlo when impacts are strongly nonlinear.

In [ ]:
from impacts import spatial_impact_delta_uncertainty
impact_delta = spatial_impact_delta_uncertainty(
    W_demo, 0.2, [1.1, 0.6], np.diag([0.0025, 0.01, 0.01, 0.01, 0.01]),
    gamma=[0.0, -0.2], confidence=0.95,
)
print('Direct / indirect / total:', impact_delta[1].direct, impact_delta[1].indirect, impact_delta[1].total)


## Spatial Durbin common-factor restriction test

Compare the unrestricted SDM likelihood with the nested SEM likelihood. The degrees of freedom count only spatially lagged covariates independently identified beyond X.

In [ ]:
from spatial_durbin_common_factor import spatial_durbin_common_factor_test
common_factor = spatial_durbin_common_factor_test(y_demo, X_demo, W_demo)
print('LR statistic, df, p-value:', common_factor.statistic,
      common_factor.degrees_of_freedom, common_factor.p_value)
print('Estimated restriction residuals:', common_factor.restriction_residuals)


## Spatial Durbin profile-likelihood interval

The profile is re-optimised over regression coefficients on [X, WX] at each candidate rho. Endpoints invert the chi-square(1) likelihood-ratio test.

In [ ]:
from spatial_mle_inference import spatial_parameter_profile_interval
sdm_rho_interval = spatial_parameter_profile_interval(
    y_demo, X_demo, W_demo, model='SDM', grid_size=121
)
print('SDM rho interval:', sdm_rho_interval.lower, sdm_rho_interval.estimate,
      sdm_rho_interval.upper)
